<a href="https://colab.research.google.com/github/mena-04/qml/blob/main/testing/quantum_vs_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install -q pennylane torch scikit-learn pandas matplotlib


In [ ]:

import numpy as np
import pandas as pd
import torch
import pennylane as qml
import matplotlib.pyplot as plt
import time
import random

from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float64)

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)


PyTorch: 2.11.0+cpu
PennyLane: 0.45.1


In [ ]:

from google.colab import files
import zipfile

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

dataset = load_breast_cancer()

X_raw = dataset.data

# sklearn: malignant=0, benign=1
# Our project: malignant=1, benign=0
y_all = (dataset.target == 0).astype(int)

with np.load(
    "limited_data/data_n40_seed42.npz"
) as data:
    train_idx = data["train_idx"].copy()

X_subset = X_raw[train_idx]
y_subset = y_all[train_idx]

print("Training subset:", X_subset.shape)
print("Malignant:", np.sum(y_subset == 1))
print("Benign:", np.sum(y_subset == 0))


Saving qbreast_limited_data.zip to qbreast_limited_data.zip
Training subset: (40, 30)
Malignant: 15
Benign: 25


In [ ]:

X_train_raw, X_val_raw, y_train, y_val = (
    train_test_split(
        X_subset,
        y_subset,
        test_size=0.25,
        stratify=y_subset,
        random_state=SEED
    )
)

preprocessor = Pipeline([
    ("standardize", StandardScaler()),
    ("pca", PCA(n_components=4)),
    ("angles", MinMaxScaler(
        feature_range=(0, np.pi),
        clip=True
    ))
])

X_train = preprocessor.fit_transform(
    X_train_raw
)

X_val = preprocessor.transform(
    X_val_raw
)

X_train_t = torch.tensor(
    X_train,
    dtype=torch.float64
)

X_val_t = torch.tensor(
    X_val,
    dtype=torch.float64
)

y_train_t = torch.tensor(
    y_train,
    dtype=torch.float64
)

y_val_t = torch.tensor(
    y_val,
    dtype=torch.float64
)

print("Training:", X_train_t.shape)
print("Validation:", X_val_t.shape)


Training: torch.Size([30, 4])
Validation: torch.Size([10, 4])


In [ ]:

N_QUBITS = 4
N_LAYERS = 2

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):

    # Encode the four diagnostic features
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)

    # Trainable variational layers
    for layer in range(N_LAYERS):

        for i in range(N_QUBITS):
            qml.RY(
                weights[layer, i, 0],
                wires=i
            )
            qml.RZ(
                weights[layer, i, 1],
                wires=i
            )

        # Entangle neighboring qubits
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])

    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]

weight_shapes = {
    "weights": (N_LAYERS, N_QUBITS, 2)
}

print(qml.draw(quantum_circuit)(
    X_train_t[0],
    torch.zeros(
        N_LAYERS, N_QUBITS, 2
    )
))


0: ──RY(2.30)──RY(0.00)──RZ(0.00)─╭●──RY(0.00)──RZ(0.00)───────────╭●──────────────┤  <Z>
1: ──RY(0.00)──RY(0.00)──RZ(0.00)─╰X─╭●─────────RY(0.00)──RZ(0.00)─╰X────────╭●────┤  <Z>
2: ──RY(1.40)──RY(0.00)──RZ(0.00)────╰X────────╭●─────────RY(0.00)──RZ(0.00)─╰X─╭●─┤  <Z>
3: ──RY(1.36)──RY(0.00)──RZ(0.00)──────────────╰X─────────RY(0.00)──RZ(0.00)────╰X─┤  <Z>


In [ ]:

import torch.nn as nn

class HybridQNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.quantum = qml.qnn.TorchLayer(
            quantum_circuit,
            weight_shapes
        )

        self.output = nn.Linear(
            N_QUBITS, 1
        )

    def forward(self, x):

        quantum_outputs = []

        for sample in x:
            q_out = self.quantum(sample)
            quantum_outputs.append(q_out)

        quantum_outputs = torch.stack(
            quantum_outputs
        )

        logits = self.output(
            quantum_outputs
        )

        return logits.squeeze(-1)


model = HybridQNN().double()

print(model)


HybridQNN(
  (quantum): <Quantum Torch Layer: func=quantum_circuit>
  (output): Linear(in_features=4, out_features=1, bias=True)
)


In [ ]:

model.train()

sample_logits = model(
    X_train_t[:4]
)

print("Output shape:", sample_logits.shape)
print("Logits:", sample_logits.detach().numpy())

assert sample_logits.shape == (4,)

criterion = nn.BCEWithLogitsLoss()

loss = criterion(
    sample_logits,
    y_train_t[:4]
)

loss.backward()

gradient_norms = {
    name: float(param.grad.norm())
    for name, param in model.named_parameters()
    if param.grad is not None
}

print("Gradient norms:")
print(gradient_norms)

assert all(
    np.isfinite(value)
    for value in gradient_norms.values()
)

assert any(
    value > 1e-10
    for name, value in gradient_norms.items()
    if "quantum" in name
)

print("QNN gradients verified.")


Output shape: torch.Size([4])
Logits: [ 0.26326501 -0.55239363 -0.68051387 -0.20446969]
Gradient norms:
{'quantum.weights': 0.10337132733865206, 'output.weight': 0.3033730741613722, 'output.bias': 0.07101136674312972}
QNN gradients verified.


In [ ]:

# Reinitialize model after gradient test
torch.manual_seed(SEED)

model = HybridQNN().double()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.03
)

criterion = nn.BCEWithLogitsLoss()

EPOCHS = 30

history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

start = time.perf_counter()

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    logits = model(X_train_t)

    loss = criterion(
        logits, y_train_t
    )

    loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():

        train_loss = loss.item()

        val_logits = model(X_val_t)

        val_loss = criterion(
            val_logits, y_val_t
        ).item()

        val_probs = torch.sigmoid(
            val_logits
        ).numpy()

        val_auc = roc_auc_score(
            y_val, val_probs
        )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_auc"].append(
        val_auc
    )

    if (
        epoch == 0
        or (epoch + 1) % 5 == 0
    ):
        print(
            f"Epoch {epoch+1:02d} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

training_seconds = (
    time.perf_counter() - start
)

print(
    "Total training time:",
    round(training_seconds, 2),
    "seconds"
)


Epoch 01 | Train loss: 0.5513 | Val loss: 0.6008 | Val AUC: 0.8333
Epoch 05 | Train loss: 0.4678 | Val loss: 0.5634 | Val AUC: 0.8333
Epoch 10 | Train loss: 0.3795 | Val loss: 0.5375 | Val AUC: 0.8333
Epoch 15 | Train loss: 0.3222 | Val loss: 0.5070 | Val AUC: 0.8750
Epoch 20 | Train loss: 0.2792 | Val loss: 0.4565 | Val AUC: 0.8750
Epoch 25 | Train loss: 0.2509 | Val loss: 0.4263 | Val AUC: 0.8750
Epoch 30 | Train loss: 0.2312 | Val loss: 0.4264 | Val AUC: 0.9167
Total training time: 23.08 seconds


In [ ]:

history_df = pd.DataFrame(history)

print(
    "Initial training loss:",
    history_df["train_loss"].iloc[0]
)

print(
    "Final training loss:",
    history_df["train_loss"].iloc[-1]
)

print(
    "Best validation AUC:",
    history_df["val_auc"].max()
)

print(
    "Final validation AUC:",
    history_df["val_auc"].iloc[-1]
)

display(history_df.tail(10).round(4))


Initial training loss: 0.5513099824186035
Final training loss: 0.23123977494075626
Best validation AUC: 0.9166666666666667
Final validation AUC: 0.9166666666666667


,train_loss,val_loss,val_auc
20,0.2725,0.4476,0.8750
21,0.2664,0.4400,0.8750
22,0.2609,0.4340,0.8750
23,0.2558,0.4294,0.8750
24,0.2509,0.4263,0.8750
25,0.2463,0.4246,0.9167
26,0.2420,0.4241,0.9167
27,0.2380,0.4245,0.9167
28,0.2344,0.4255,0.9167
29,0.2312,0.4264,0.9167


In [ ]:

OUTPUT_DIR = Path("phase_6_results")
OUTPUT_DIR.mkdir(exist_ok=True)

history_df.to_csv(
    OUTPUT_DIR / "qnn_training_history.csv",
    index=False
)

torch.save(
    model.state_dict(),
    OUTPUT_DIR / "qnn_initial_model.pt"
)

print("Phase 6.1 results saved.")


Phase 6.1 results saved.


In [ ]:

import torch.nn as nn

class ClassicalMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 4),
            nn.Tanh(),
            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


torch.manual_seed(SEED)

mlp = ClassicalMLP().double()

print("Classical MLP:")
print(mlp)

print(
    "Trainable parameters:",
    sum(p.numel() for p in mlp.parameters())
)


Classical MLP:
ClassicalMLP(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=4, bias=True)
    (1): Tanh()
    (2): Linear(in_features=4, out_features=1, bias=True)
  )
)
Trainable parameters: 25


In [ ]:

torch.manual_seed(SEED)

mlp = ClassicalMLP().double()

mlp_optimizer = torch.optim.Adam(
    mlp.parameters(),
    lr=0.03
)

criterion = nn.BCEWithLogitsLoss()

mlp_history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

start = time.perf_counter()

for epoch in range(30):

    mlp.train()
    mlp_optimizer.zero_grad()

    logits = mlp(X_train_t)

    loss = criterion(logits, y_train_t)

    loss.backward()
    mlp_optimizer.step()

    mlp.eval()

    with torch.no_grad():

        val_logits = mlp(X_val_t)

        val_loss = criterion(
            val_logits, y_val_t
        ).item()

        val_probs = torch.sigmoid(
            val_logits
        ).numpy()

        val_auc = roc_auc_score(
            y_val, val_probs
        )

    mlp_history["train_loss"].append(
        loss.item()
    )

    mlp_history["val_loss"].append(
        val_loss
    )

    mlp_history["val_auc"].append(
        val_auc
    )

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:02d} | "
            f"Train loss: {loss.item():.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

mlp_training_seconds = (
    time.perf_counter() - start
)

print(
    "MLP training time:",
    round(mlp_training_seconds, 4),
    "seconds"
)


Epoch 01 | Train loss: 0.7164 | Val loss: 0.6931 | Val AUC: 0.4583
Epoch 05 | Train loss: 0.6540 | Val loss: 0.6724 | Val AUC: 0.6667
Epoch 10 | Train loss: 0.6434 | Val loss: 0.6742 | Val AUC: 0.8333
Epoch 15 | Train loss: 0.5915 | Val loss: 0.6425 | Val AUC: 0.8750
Epoch 20 | Train loss: 0.5050 | Val loss: 0.5965 | Val AUC: 0.9167
Epoch 25 | Train loss: 0.4061 | Val loss: 0.5311 | Val AUC: 1.0000
Epoch 30 | Train loss: 0.3130 | Val loss: 0.4722 | Val AUC: 1.0000
MLP training time: 0.2679 seconds


In [ ]:

def evaluate_neural_model(model, X, y):
    model.eval()

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits).numpy()

    predictions = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, predictions, labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(y, probs),
        "recall": recall_score(y, predictions),
        "precision": precision_score(
            y, predictions, zero_division=0
        ),
        "f1": f1_score(
            y, predictions, zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "false_negatives": int(fn),
        "false_positives": int(fp)
    }


neural_comparison = pd.DataFrame([
    {
        "model": "Hybrid QNN",
        "parameters": 21,
        "training_seconds": training_seconds,
        **evaluate_neural_model(
            model, X_val_t, y_val
        )
    },
    {
        "model": "Classical MLP",
        "parameters": 25,
        "training_seconds": mlp_training_seconds,
        **evaluate_neural_model(
            mlp, X_val_t, y_val
        )
    }
])

display(neural_comparison.round(4))


,model,parameters,training_seconds,roc_auc,recall,precision,f1,specificity,false_negatives,false_positives
0,Hybrid QNN,21,23.0806,0.9167,0.5,1.0,0.6667,1.0,2,0
1,Classical MLP,25,0.2679,1.0000,0.5,1.0,0.6667,1.0,2,0


In [ ]:

class ConfigurableQNN(nn.Module):

    def __init__(self, n_layers=1, entangle=True):
        super().__init__()

        self.n_layers = n_layers
        self.entangle = entangle

        dev = qml.device(
            "default.qubit",
            wires=4
        )

        @qml.qnode(
            dev,
            interface="torch",
            diff_method="backprop"
        )
        def circuit(inputs, weights):

            for i in range(4):
                qml.RY(inputs[i], wires=i)

            for layer in range(n_layers):

                for i in range(4):
                    qml.RY(
                        weights[layer, i, 0],
                        wires=i
                    )
                    qml.RZ(
                        weights[layer, i, 1],
                        wires=i
                    )

                if entangle:
                    for i in range(3):
                        qml.CNOT(
                            wires=[i, i + 1]
                        )

            return [
                qml.expval(qml.PauliZ(i))
                for i in range(4)
            ]

        self.quantum = qml.qnn.TorchLayer(
            circuit,
            {
                "weights": (n_layers, 4, 2)
            }
        )

        self.output = nn.Linear(4, 1)

    def forward(self, x):

        outputs = torch.stack([
            self.quantum(sample)
            for sample in x
        ])

        return self.output(outputs).squeeze(-1)


In [ ]:

ARCHITECTURES = {
    "QNN-A: 1 layer, no entanglement": (1, False),
    "QNN-B: 1 layer, entangled": (1, True),
    "QNN-C: 2 layers, entangled": (2, True)
}

architecture_records = []
architecture_histories = {}

for name, (layers, entangle) in ARCHITECTURES.items():

    torch.manual_seed(SEED)

    candidate = ConfigurableQNN(
        n_layers=layers,
        entangle=entangle
    ).double()

    optimizer = torch.optim.Adam(
        candidate.parameters(),
        lr=0.03
    )

    criterion = nn.BCEWithLogitsLoss()

    losses = []

    start = time.perf_counter()

    for epoch in range(30):

        candidate.train()
        optimizer.zero_grad()

        logits = candidate(X_train_t)
        loss = criterion(logits, y_train_t)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    elapsed = time.perf_counter() - start

    metrics = evaluate_neural_model(
        candidate,
        X_val_t,
        y_val
    )

    architecture_records.append({
        "architecture": name,
        "layers": layers,
        "entanglement": entangle,
        "parameters": sum(
            p.numel()
            for p in candidate.parameters()
        ),
        "initial_loss": losses[0],
        "final_loss": losses[-1],
        "training_seconds": elapsed,
        **metrics
    })

    architecture_histories[name] = losses

architecture_df = pd.DataFrame(
    architecture_records
)

display(architecture_df.round(4))


,architecture,layers,entanglement,parameters,initial_loss,final_loss,training_seconds,roc_auc,recall,precision,f1,specificity,false_negatives,false_positives
0,"QNN-A: 1 layer, no entanglement",1,False,13,0.7092,0.3839,8.7448,0.9167,0.25,1.0,0.4000,1.0,3,0
1,"QNN-B: 1 layer, entangled",1,True,13,0.6911,0.4118,10.4605,0.8333,0.25,1.0,0.4000,1.0,3,0
2,"QNN-C: 2 layers, entangled",2,True,21,0.5513,0.2312,15.0638,0.9167,0.50,1.0,0.6667,1.0,2,0


In [ ]:

from pathlib import Path

OUTPUT_DIR = Path("phase_6_results")
OUTPUT_DIR.mkdir(exist_ok=True)

architecture_df.to_csv(
    OUTPUT_DIR / "qnn_architecture_ablation.csv",
    index=False
)

pd.DataFrame(
    architecture_histories
).to_csv(
    OUTPUT_DIR / "qnn_architecture_losses.csv",
    index=False
)

print("Phase 6.3 results saved.")


Phase 6.3 results saved.


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
import time
import zipfile

from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

P64_SIZES = [40, 80, 160]
P64_SEEDS = [42, 123, 2026]
P64_EPOCHS = 30
P64_LR = 0.03

P64_MODELS = [
    "Classical MLP",
    "QNN-A",
    "QNN-B",
    "QNN-C"
]

P64_DIR = Path("phase_6_results")
P64_DIR.mkdir(exist_ok=True)

torch.set_default_dtype(torch.float64)

dataset = load_breast_cancer()
X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

print("Phase 6.4 ready.")


Phase 6.4 ready.


In [ ]:

from google.colab import files

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

for n in P64_SIZES:
    for seed in P64_SEEDS:
        path = Path(
            f"limited_data/data_n{n}_seed{seed}.npz"
        )
        assert path.exists(), f"Missing: {path}"

print("All nine training subsets available.")


All nine training subsets available.


In [ ]:

def p64_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])


def p64_make_model(model_name):

    if model_name == "Classical MLP":
        return ClassicalMLP().double()

    if model_name == "QNN-A":
        return ConfigurableQNN(
            n_layers=1,
            entangle=False
        ).double()

    if model_name == "QNN-B":
        return ConfigurableQNN(
            n_layers=1,
            entangle=True
        ).double()

    if model_name == "QNN-C":
        return ConfigurableQNN(
            n_layers=2,
            entangle=True
        ).double()

    raise ValueError(model_name)


In [ ]:

def p64_train_evaluate(
    model_name,
    Xtr,
    ytr,
    Xval,
    yval,
    seed
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = p64_make_model(model_name)

    Xtr_t = torch.tensor(
        Xtr, dtype=torch.float64
    )
    Xval_t = torch.tensor(
        Xval, dtype=torch.float64
    )

    ytr_t = torch.tensor(
        ytr, dtype=torch.float64
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=P64_LR
    )

    criterion = nn.BCEWithLogitsLoss()

    start = time.perf_counter()

    for epoch in range(P64_EPOCHS):

        model.train()
        optimizer.zero_grad()

        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)

        loss.backward()
        optimizer.step()

    training_seconds = (
        time.perf_counter() - start
    )

    # Evaluate after final parameter update
    model.eval()

    with torch.no_grad():
        logits_val = model(Xval_t)
        probabilities = (
            torch.sigmoid(logits_val)
            .cpu().numpy()
        )

        final_train_loss = criterion(
            model(Xtr_t), ytr_t
        ).item()

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yval,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            yval, probabilities
        ),
        "recall": recall_score(
            yval, predictions,
            zero_division=0
        ),
        "precision": precision_score(
            yval, predictions,
            zero_division=0
        ),
        "f1": f1_score(
            yval, predictions,
            zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "final_train_loss": final_train_loss,
        "training_seconds": training_seconds,
        "parameters": sum(
            p.numel()
            for p in model.parameters()
        )
    }


In [ ]:

p64_records = []

for n_samples in P64_SIZES:
    for seed in P64_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            prep = p64_preprocessor()

            Xtr = prep.fit_transform(
                X_subset[tr]
            )

            Xval = prep.transform(
                X_subset[val]
            )

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P64_MODELS:

                result = p64_train_evaluate(
                    model_name,
                    Xtr,
                    ytr,
                    Xval,
                    yval,
                    seed=seed + fold
                )

                p64_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    **result
                })

                print(
                    f"n={n_samples}, "
                    f"seed={seed}, "
                    f"fold={fold}, "
                    f"{model_name}: "
                    f"AUC={result['roc_auc']:.4f}, "
                    f"Recall={result['recall']:.4f}"
                )

        # Save progress after every seed
        pd.DataFrame(p64_records).to_csv(
            P64_DIR / "phase64_raw.csv",
            index=False
        )

p64_df = pd.DataFrame(p64_records)

print("Phase 6.4 completed.")


n=40, seed=42, fold=0, Classical MLP: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=0, QNN-A: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=0, QNN-B: AUC=0.8889, Recall=0.8000
n=40, seed=42, fold=0, QNN-C: AUC=0.9778, Recall=0.6000
n=40, seed=42, fold=1, Classical MLP: AUC=0.8250, Recall=0.6000
n=40, seed=42, fold=1, QNN-A: AUC=0.8750, Recall=0.4000
n=40, seed=42, fold=1, QNN-B: AUC=0.9500, Recall=0.8000
n=40, seed=42, fold=1, QNN-C: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=2, Classical MLP: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=2, QNN-A: AUC=1.0000, Recall=0.6000
n=40, seed=42, fold=2, QNN-B: AUC=1.0000, Recall=0.6000
n=40, seed=42, fold=2, QNN-C: AUC=0.9750, Recall=1.0000
n=40, seed=123, fold=0, Classical MLP: AUC=0.9556, Recall=0.6000
n=40, seed=123, fold=0, QNN-A: AUC=0.9778, Recall=0.4000
n=40, seed=123, fold=0, QNN-B: AUC=1.0000, Recall=0.6000
n=40, seed=123, fold=0, QNN-C: AUC=1.0000, Recall=0.6000
n=40, seed=123, fold=1, Classical MLP: AUC=0.8750, Recall=0.6000
n=4

In [ ]:

p64_per_seed = (
    p64_df
    .groupby([
        "n_samples",
        "seed",
        "model"
    ])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        specificity=("specificity", "mean"),
        runtime=("training_seconds", "sum")
    )
    .reset_index()
)

p64_summary = (
    p64_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        specificity_mean=("specificity", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p64_summary.round(4))

p64_per_seed.to_csv(
    P64_DIR / "phase64_per_seed.csv",
    index=False
)

p64_summary.to_csv(
    P64_DIR / "phase64_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,recall_std,precision_mean,f1_mean,specificity_mean,runtime_mean
0,40,Classical MLP,0.9534,0.0425,0.7333,0.0000,0.9722,0.8228,0.9861,0.1334
1,40,QNN-A,0.9349,0.0439,0.5778,0.0385,0.9722,0.7121,0.9877,28.6235
2,40,QNN-B,0.9198,0.0534,0.6444,0.1540,0.9185,0.7444,0.9614,26.9461
3,40,QNN-C,0.9630,0.0037,0.7556,0.0385,0.8926,0.7978,0.9321,42.4413
4,80,Classical MLP,0.9624,0.0272,0.7667,0.1000,0.9620,0.8352,0.9734,0.0822
5,80,QNN-A,0.9427,0.0218,0.5333,0.1000,1.0000,0.6633,1.0000,49.0433
6,80,QNN-B,0.9522,0.0251,0.6333,0.0667,1.0000,0.7623,1.0000,56.7810
7,80,QNN-C,0.9677,0.0141,0.7222,0.0192,0.9277,0.8023,0.9661,85.2002
8,160,Classical MLP,0.9862,0.0093,0.7722,0.0694,0.9942,0.8610,0.9966,0.1002
9,160,QNN-A,0.9609,0.0253,0.4944,0.1110,0.9926,0.6452,0.9967,95.8195


In [ ]:

paired = p64_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

comparison_records = []

for n_samples in P64_SIZES:

    subset = paired[
        paired["n_samples"] == n_samples
    ]

    for qnn in ["QNN-A", "QNN-B", "QNN-C"]:

        row = {
            "n_samples": n_samples,
            "comparison": f"{qnn} - Classical MLP"
        }

        for metric in [
            "roc_auc", "recall", "f1"
        ]:

            differences = (
                subset[f"{metric}__{qnn}"]
                - subset[f"{metric}__Classical MLP"]
            )

            row[f"{metric}_diff_mean"] = (
                differences.mean()
            )

            row[f"{metric}_diff_std"] = (
                differences.std()
            )

        comparison_records.append(row)

p64_differences = pd.DataFrame(
    comparison_records
)

display(p64_differences.round(4))

p64_differences.to_csv(
    P64_DIR / "phase64_differences.csv",
    index=False
)


,n_samples,comparison,roc_auc_diff_mean,roc_auc_diff_std,recall_diff_mean,recall_diff_std,f1_diff_mean,f1_diff_std
0,40,QNN-A - Classical MLP,-0.0185,0.0753,-0.1556,0.1333,-0.1107,0.0963
1,40,QNN-B - Classical MLP,-0.0336,0.1137,-0.0889,0.2472,-0.0784,0.2165
2,40,QNN-C - Classical MLP,0.0096,0.0582,0.0222,0.1202,-0.0249,0.0882
3,80,QNN-A - Classical MLP,-0.0197,0.0374,-0.2333,0.2179,-0.1719,0.2006
4,80,QNN-B - Classical MLP,-0.0102,0.0653,-0.1333,0.2236,-0.0728,0.1644
5,80,QNN-C - Classical MLP,0.0053,0.0624,-0.0444,0.1667,-0.0328,0.1479
6,160,QNN-A - Classical MLP,-0.0253,0.0304,-0.2778,0.1481,-0.2158,0.1259
7,160,QNN-B - Classical MLP,-0.0771,0.1108,-0.2667,0.3335,-0.2896,0.3491
8,160,QNN-C - Classical MLP,-0.0179,0.0183,-0.0333,0.1250,-0.0346,0.0915


In [ ]:

complexity_summary = (
    p64_df
    .groupby("model")
    .agg(
        parameters=("parameters", "first"),
        mean_training_seconds=(
            "training_seconds", "mean"
        ),
        std_training_seconds=(
            "training_seconds", "std"
        ),
        mean_final_loss=(
            "final_train_loss", "mean"
        )
    )
    .reset_index()
)

display(complexity_summary.round(4))

complexity_summary.to_csv(
    P64_DIR / "phase64_complexity.csv",
    index=False
)


,model,parameters,mean_training_seconds,std_training_seconds,mean_final_loss
0,Classical MLP,25,0.0351,0.0223,0.3319
1,QNN-A,13,19.2763,9.7128,0.4646
2,QNN-B,13,21.3834,11.5690,0.4199
3,QNN-C,21,33.1646,18.2062,0.3354


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
import time
import json

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

P65_SIZES = [40, 80]
P65_SEEDS = [42, 123, 2026]

P65_EPOCHS = [30, 60, 100]
P65_LRS = [0.01, 0.03]
P65_INIT_SEEDS = [42, 123, 2026]

P65_MODELS = [
    "Classical MLP",
    "QNN-C"
]

P65_DIR = Path("phase_6_results")
P65_DIR.mkdir(exist_ok=True)

torch.set_default_dtype(torch.float64)

print("Phase 6.5 ready.")


Phase 6.5 ready.


In [ ]:

def p65_train_model(
    model_name,
    Xtr,
    ytr,
    Xval,
    yval,
    epochs,
    lr,
    init_seed
):
    random.seed(init_seed)
    np.random.seed(init_seed)
    torch.manual_seed(init_seed)

    model = p64_make_model(model_name)

    Xtr_t = torch.tensor(
        Xtr, dtype=torch.float64
    )
    Xval_t = torch.tensor(
        Xval, dtype=torch.float64
    )
    ytr_t = torch.tensor(
        ytr, dtype=torch.float64
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.BCEWithLogitsLoss()

    start = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)

        loss.backward()
        optimizer.step()

    elapsed = time.perf_counter() - start

    model.eval()

    with torch.no_grad():
        val_logits = model(Xval_t)
        val_probs = torch.sigmoid(
            val_logits
        ).cpu().numpy()

    pred = (val_probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yval, pred, labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(yval, val_probs),
        "recall": recall_score(
            yval, pred, zero_division=0
        ),
        "precision": precision_score(
            yval, pred, zero_division=0
        ),
        "f1": f1_score(
            yval, pred, zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "training_seconds": elapsed,
        "false_negatives": int(fn),
        "false_positives": int(fp)
    }


In [ ]:

def p65_select_config(
    X_raw_train,
    y_train,
    model_name,
    seed
):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    records = []

    for epochs in P65_EPOCHS:
        for lr in P65_LRS:

            fold_aucs = []

            for fold, (tr, val) in enumerate(
                inner_cv.split(
                    X_raw_train, y_train
                )
            ):

                prep = p64_preprocessor()

                Xtr = prep.fit_transform(
                    X_raw_train[tr]
                )

                Xval = prep.transform(
                    X_raw_train[val]
                )

                result = p65_train_model(
                    model_name,
                    Xtr,
                    y_train[tr],
                    Xval,
                    y_train[val],
                    epochs,
                    lr,
                    init_seed=seed + fold
                )

                fold_aucs.append(
                    result["roc_auc"]
                )

            records.append({
                "epochs": epochs,
                "lr": lr,
                "cv_auc": float(
                    np.mean(fold_aucs)
                )
            })

    results = pd.DataFrame(records)

    best = results.sort_values(
        ["cv_auc", "epochs", "lr"],
        ascending=[False, True, True]
    ).iloc[0]

    return {
        "epochs": int(best["epochs"]),
        "lr": float(best["lr"])
    }, results


In [ ]:

from pathlib import Path

with np.load(
    "limited_data/data_n40_seed42.npz"
) as data:
    subset_idx = data["train_idx"].copy()

X_pilot = X_raw[subset_idx]
y_pilot = y_all[subset_idx]

# One outer fold for a pilot
pilot_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

pilot_tr, pilot_val = next(
    pilot_cv.split(X_pilot, y_pilot)
)

pilot_records = []

for model_name in P65_MODELS:

    start = time.perf_counter()

    config, search_results = p65_select_config(
        X_pilot[pilot_tr],
        y_pilot[pilot_tr],
        model_name,
        seed=42
    )

    elapsed = time.perf_counter() - start

    print("\nModel:", model_name)
    print("Selected:", config)
    print("Search time:", round(elapsed, 2), "seconds")

    display(search_results.round(4))

    pilot_records.append({
        "model": model_name,
        "config": config,
        "search_seconds": elapsed
    })

pilot_df = pd.DataFrame(pilot_records)



Model: Classical MLP
Selected: {'epochs': 100, 'lr': 0.03}
Search time: 2.73 seconds


,epochs,lr,cv_auc
0,30,0.01,0.8611
1,30,0.03,0.8019
2,60,0.01,0.8019
3,60,0.03,0.7963
4,100,0.01,0.8352
5,100,0.03,0.8722



Model: QNN-C
Selected: {'epochs': 30, 'lr': 0.03}
Search time: 350.52 seconds


,epochs,lr,cv_auc
0,30,0.01,0.8500
1,30,0.03,0.9167
2,60,0.01,0.9167
3,60,0.03,0.9167
4,100,0.01,0.9167
5,100,0.03,0.9167


In [ ]:

P65_INIT_SEEDS = [42, 123, 2026]

stability_records = []

# Use the same 40-sample subset as the pilot
with np.load(
    "limited_data/data_n40_seed42.npz"
) as data:
    subset_idx = data["train_idx"].copy()

X_stability = X_raw[subset_idx]
y_stability = y_all[subset_idx]

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

for fold, (tr, val) in enumerate(
    cv.split(X_stability, y_stability)
):

    prep = p64_preprocessor()

    Xtr = prep.fit_transform(
        X_stability[tr]
    )

    Xval = prep.transform(
        X_stability[val]
    )

    for model_name in ["Classical MLP", "QNN-C"]:

        for init_seed in P65_INIT_SEEDS:

            result = p65_train_model(
                model_name,
                Xtr,
                y_stability[tr],
                Xval,
                y_stability[val],
                epochs=30,
                lr=0.03,
                init_seed=init_seed
            )

            stability_records.append({
                "fold": fold,
                "model": model_name,
                "init_seed": init_seed,
                **result
            })

            print(
                f"Fold {fold}, {model_name}, "
                f"seed={init_seed}: "
                f"AUC={result['roc_auc']:.4f}"
            )

stability_df = pd.DataFrame(stability_records)

display(stability_df.round(4))


Fold 0, Classical MLP, seed=42: AUC=1.0000
Fold 0, Classical MLP, seed=123: AUC=1.0000
Fold 0, Classical MLP, seed=2026: AUC=0.9778
Fold 0, QNN-C, seed=42: AUC=0.9778
Fold 0, QNN-C, seed=123: AUC=0.9778
Fold 0, QNN-C, seed=2026: AUC=0.9556
Fold 1, Classical MLP, seed=42: AUC=0.8500
Fold 1, Classical MLP, seed=123: AUC=0.8750
Fold 1, Classical MLP, seed=2026: AUC=0.8500
Fold 1, QNN-C, seed=42: AUC=0.9500
Fold 1, QNN-C, seed=123: AUC=0.9250
Fold 1, QNN-C, seed=2026: AUC=0.8750
Fold 2, Classical MLP, seed=42: AUC=0.9250
Fold 2, Classical MLP, seed=123: AUC=0.9500
Fold 2, Classical MLP, seed=2026: AUC=0.8500
Fold 2, QNN-C, seed=42: AUC=0.9750
Fold 2, QNN-C, seed=123: AUC=0.9750
Fold 2, QNN-C, seed=2026: AUC=0.9750


,fold,model,init_seed,roc_auc,recall,precision,f1,specificity,training_seconds,false_negatives,false_positives
0,0,Classical MLP,42,1.0000,0.8,1.0000,0.8889,1.0000,0.0442,1,0
1,0,Classical MLP,123,1.0000,0.8,1.0000,0.8889,1.0000,0.0479,1,0
2,0,Classical MLP,2026,0.9778,1.0,0.8333,0.9091,0.8889,0.1516,0,1
3,0,QNN-C,42,0.9778,0.6,1.0000,0.7500,1.0000,19.9482,2,0
4,0,QNN-C,123,0.9778,0.6,1.0000,0.7500,1.0000,13.1168,2,0
5,0,QNN-C,2026,0.9556,0.8,1.0000,0.8889,1.0000,13.5246,1,0
6,1,Classical MLP,42,0.8500,0.6,1.0000,0.7500,1.0000,0.0254,2,0
7,1,Classical MLP,123,0.8750,0.6,1.0000,0.7500,1.0000,0.0263,2,0
8,1,Classical MLP,2026,0.8500,0.6,1.0000,0.7500,1.0000,0.0282,2,0
9,1,QNN-C,42,0.9500,0.8,1.0000,0.8889,1.0000,14.7879,1,0


In [ ]:

# First average across validation folds
per_initialization = (
    stability_df
    .groupby(["model", "init_seed"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        runtime=("training_seconds", "sum")
    )
    .reset_index()
)

# Then summarize across initialization seeds
stability_summary = (
    per_initialization
    .groupby("model")
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(stability_summary.round(4))

stability_df.to_csv(
    P65_DIR / "phase65_initialization_raw.csv",
    index=False
)

stability_summary.to_csv(
    P65_DIR / "phase65_initialization_summary.csv",
    index=False
)


,model,auc_mean,auc_std,recall_mean,recall_std,f1_mean,runtime_mean
0,Classical MLP,0.9198,0.0250,0.7556,0.0385,0.835,0.1462
1,QNN-C,0.9540,0.0168,0.8222,0.0385,0.845,45.2975


In [ ]:

def p66_train_checkpoints(
    model_name,
    Xtr, ytr,
    Xval, yval,
    lr,
    init_seed
):
    random.seed(init_seed)
    np.random.seed(init_seed)
    torch.manual_seed(init_seed)

    model = p64_make_model(model_name)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float64)
    Xval_t = torch.tensor(Xval, dtype=torch.float64)
    ytr_t = torch.tensor(ytr, dtype=torch.float64)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.BCEWithLogitsLoss()

    checkpoints = P66_EPOCHS[model_name]
    max_epochs = max(checkpoints)

    results = {}
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):

        model.train()
        optimizer.zero_grad()

        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)

        loss.backward()
        optimizer.step()

        if epoch in checkpoints:

            model.eval()

            with torch.no_grad():
                val_logits = model(Xval_t)
                probabilities = torch.sigmoid(
                    val_logits
                ).cpu().numpy()

            predictions = (
                probabilities >= 0.5
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                yval,
                predictions,
                labels=[0, 1]
            ).ravel()

            results[epoch] = {
                "roc_auc": roc_auc_score(
                    yval, probabilities
                ),
                "recall": recall_score(
                    yval, predictions,
                    zero_division=0
                ),
                "precision": precision_score(
                    yval, predictions,
                    zero_division=0
                ),
                "f1": f1_score(
                    yval, predictions,
                    zero_division=0
                ),
                "specificity": tn / (tn + fp),
                "elapsed_seconds": (
                    time.perf_counter() - start
                )
            }

    return results


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
import time
import json

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

P66_SIZES = [40, 80]
P66_SEEDS = [42, 123, 2026]

P66_LRS = [0.01, 0.03]

P66_EPOCHS = {
    "Classical MLP": [30, 60, 100],
    "QNN-C": [30, 60]
}

P66_MODELS = [
    "Classical MLP",
    "QNN-C"
]

P66_DIR = Path("phase_6_results")
P66_DIR.mkdir(exist_ok=True)

torch.set_default_dtype(torch.float64)

print("Phase 6.6 configuration ready.")


Phase 6.6 configuration ready.


In [ ]:

def p66_select_config(
    X_train_raw,
    y_train,
    model_name,
    seed
):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    records = []

    for fold, (tr, val) in enumerate(
        inner_cv.split(
            X_train_raw,
            y_train
        )
    ):

        prep = p64_preprocessor()

        Xtr = prep.fit_transform(
            X_train_raw[tr]
        )

        Xval = prep.transform(
            X_train_raw[val]
        )

        for lr in P66_LRS:

            checkpoint_results = p66_train_checkpoints(
                model_name,
                Xtr,
                y_train[tr],
                Xval,
                y_train[val],
                lr,
                init_seed=seed + fold
            )

            for epochs, metrics in (
                checkpoint_results.items()
            ):

                records.append({
                    "fold": fold,
                    "epochs": epochs,
                    "lr": lr,
                    "roc_auc": metrics["roc_auc"]
                })

    search_df = pd.DataFrame(records)

    summary = (
        search_df
        .groupby(["epochs", "lr"])
        ["roc_auc"]
        .mean()
        .reset_index(name="mean_cv_auc")
    )

    # Prefer fewer epochs when CV AUC ties
    summary = summary.sort_values(
        ["mean_cv_auc", "epochs", "lr"],
        ascending=[False, True, True]
    )

    best = summary.iloc[0]

    return {
        "epochs": int(best["epochs"]),
        "lr": float(best["lr"])
    }, summary


In [ ]:

def p66_evaluate_selected(
    model_name,
    Xtr_raw,
    ytr,
    Xval_raw,
    yval,
    config,
    seed
):
    prep = p64_preprocessor()

    Xtr = prep.fit_transform(Xtr_raw)
    Xval = prep.transform(Xval_raw)

    checkpoints = p66_train_checkpoints(
        model_name,
        Xtr, ytr,
        Xval, yval,
        lr=config["lr"],
        init_seed=seed
    )

    # The helper trains through max_epochs.
    # For selected shorter epochs, its saved metrics
    # reflect the earlier checkpoint.
    return checkpoints[config["epochs"]]


In [42]:

p66_records = []
p66_search_records = []

for n_samples in P66_SIZES:
    for seed in P66_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr_raw = X_subset[tr]
            Xval_raw = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P66_MODELS:

                start = time.perf_counter()

                config, search_summary = (
                    p66_select_config(
                        Xtr_raw,
                        ytr,
                        model_name,
                        seed + fold
                    )
                )

                selection_seconds = (
                    time.perf_counter() - start
                )

                metrics = p66_evaluate_selected(
                    model_name,
                    Xtr_raw,
                    ytr,
                    Xval_raw,
                    yval,
                    config,
                    seed + fold
                )

                p66_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    "config": json.dumps(config),
                    "selection_seconds": selection_seconds,
                    **metrics
                })

                for _, row in search_summary.iterrows():
                    p66_search_records.append({
                        "n_samples": n_samples,
                        "seed": seed,
                        "fold": fold,
                        "model": model_name,
                        "epochs": int(row["epochs"]),
                        "lr": float(row["lr"]),
                        "inner_cv_auc": row["mean_cv_auc"]
                    })

                print(
                    f"n={n_samples}, "
                    f"seed={seed}, "
                    f"fold={fold}, "
                    f"{model_name}: "
                    f"selected={config}, "
                    f"AUC={metrics['roc_auc']:.4f}"
                )

        pd.DataFrame(p66_records).to_csv(
            P66_DIR / "phase66_raw.csv",
            index=False
        )

        pd.DataFrame(
            p66_search_records
        ).to_csv(
            P66_DIR / "phase66_search.csv",
            index=False
        )

p66_df = pd.DataFrame(p66_records)

print("Phase 6.6 completed.")


n=40, seed=42, fold=0, Classical MLP: selected={'epochs': 100, 'lr': 0.03}, AUC=1.0000
n=40, seed=42, fold=0, QNN-C: selected={'epochs': 30, 'lr': 0.03}, AUC=0.9778
n=40, seed=42, fold=1, Classical MLP: selected={'epochs': 100, 'lr': 0.03}, AUC=0.9250
n=40, seed=42, fold=1, QNN-C: selected={'epochs': 30, 'lr': 0.01}, AUC=0.9250
n=40, seed=42, fold=2, Classical MLP: selected={'epochs': 60, 'lr': 0.03}, AUC=0.8750
n=40, seed=42, fold=2, QNN-C: selected={'epochs': 60, 'lr': 0.03}, AUC=1.0000
n=40, seed=123, fold=0, Classical MLP: selected={'epochs': 60, 'lr': 0.03}, AUC=0.9556
n=40, seed=123, fold=0, QNN-C: selected={'epochs': 30, 'lr': 0.03}, AUC=1.0000
n=40, seed=123, fold=1, Classical MLP: selected={'epochs': 60, 'lr': 0.03}, AUC=0.9500
n=40, seed=123, fold=1, QNN-C: selected={'epochs': 60, 'lr': 0.03}, AUC=0.9000
n=40, seed=123, fold=2, Classical MLP: selected={'epochs': 30, 'lr': 0.03}, AUC=1.0000
n=40, seed=123, fold=2, QNN-C: selected={'epochs': 60, 'lr': 0.03}, AUC=1.0000
n=40, se

In [45]:

p66_per_seed = (
    p66_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        specificity=("specificity", "mean"),
        selection_seconds=("selection_seconds", "sum"),
        training_seconds=("elapsed_seconds", "sum")
    )
    .reset_index()
)

p66_summary = (
    p66_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        specificity_mean=("specificity", "mean"),
        selection_time_mean=("selection_seconds", "mean"),
        training_time_mean=("training_seconds", "mean")
    )
    .reset_index()
)

display(p66_summary.round(4))

p66_summary.to_csv(
    P66_DIR / "phase66_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,f1_mean,precision_mean,specificity_mean,selection_time_mean,training_time_mean
0,40,Classical MLP,0.9596,0.0231,0.7333,0.7664,0.8259,0.9614,2.6560,0.2877
1,40,QNN-C,0.9657,0.0024,0.7778,0.8089,0.8997,0.9321,341.3047,55.1615
2,80,Classical MLP,0.9848,0.0124,0.6778,0.7652,0.9889,0.9931,2.5026,0.2264
3,80,QNN-C,0.9740,0.0156,0.8000,0.8544,0.9399,0.9661,684.6842,142.2742


In [43]:

paired = p66_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

for metric in ["roc_auc", "recall", "f1"]:
    paired[f"{metric}_difference"] = (
        paired[f"{metric}__QNN-C"]
        - paired[f"{metric}__Classical MLP"]
    )

p66_differences = (
    paired
    .groupby("n_samples")
    [[
        "roc_auc_difference",
        "recall_difference",
        "f1_difference"
    ]]
    .agg(["mean", "std"])
)

display(p66_differences.round(4))


roc_auc_difference         recall_difference         f1_difference  \
                        mean     std              mean     std          mean   
n_samples                                                                      
40                    0.0062  0.0563            0.0444  0.2186        0.0425   
80                   -0.0108  0.0333            0.1222  0.2539        0.0893   

                   
              std  
n_samples          
40         0.2074  
80         0.2453

In [44]:

config_counts = (
    p66_df
    .groupby(["n_samples", "model", "config"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["n_samples", "model", "count"],
        ascending=[True, True, False]
    )
)

display(config_counts)

config_counts.to_csv(
    P66_DIR / "phase66_selected_configs.csv",
    index=False
)


,n_samples,model,config,count
4,40,Classical MLP,"{""epochs"": 60, ""lr"": 0.03}",3
0,40,Classical MLP,"{""epochs"": 100, ""lr"": 0.01}",2
1,40,Classical MLP,"{""epochs"": 100, ""lr"": 0.03}",2
2,40,Classical MLP,"{""epochs"": 30, ""lr"": 0.01}",1
3,40,Classical MLP,"{""epochs"": 30, ""lr"": 0.03}",1
6,40,QNN-C,"{""epochs"": 30, ""lr"": 0.03}",5
7,40,QNN-C,"{""epochs"": 60, ""lr"": 0.03}",3
5,40,QNN-C,"{""epochs"": 30, ""lr"": 0.01}",1
11,80,Classical MLP,"{""epochs"": 60, ""lr"": 0.03}",4
8,80,Classical MLP,"{""epochs"": 100, ""lr"": 0.03}",2
